In [1]:
import numpy as np 
import sys, os
sys.path.append('../Netket/')
import netket as nk
from jax import numpy as jnp
import itertools
from scipy.special import comb
from jax import jit, vmap
import jax
import matplotlib.pyplot as plt 
from cluster_expansion import fwht_coeffs_in_cluster_col_order, compress_and_reconstruct_cached
import sample_hams
import importlib
importlib.reload(sample_hams)
from sample_hams import get_TFI_Hamiltonian
import optimization
importlib.reload(optimization)


∣NK⟩ Tip: You can plot data with JsonLog.data['Energy'][:-30].plot().

<module 'optimization' from '/scratch/ashankar/DataMiningRBMs/ClusterExpansion/../Netket/optimization.py'>

In [2]:
nspins = 8
hilbert = nk.hilbert.Spin(s=0.5, N=nspins)
dimH  = hilbert.n_states
wf = np.random.rand(hilbert.n_states) + 1j * np.random.rand(hilbert.n_states)
wf = np.exp(1j * 4*np.pi*np.random.rand(hilbert.n_states))
# wf[dimH//2:] = 0.0 + 1e-14*1j
wf /= np.linalg.norm(wf)
logwf = np.log(wf)
logwf = (1j * 4*np.pi*np.random.rand(hilbert.n_states))
randintlist = np.random.randint(0, 10, size=hilbert.n_states)
wf = np.exp(logwf)
logwf2 = np.log(wf)
### Cluster uniqueness test
coeffs = fwht_coeffs_in_cluster_col_order(logwf, hilbert)
gauged_coeffs = coeffs  + 1j * 2*np.pi*randintlist #perform a gauge transformation occuring from compelx Log
recon = compress_and_reconstruct_cached(coeffs, len(coeffs), hilbert)
recond_gauged = compress_and_reconstruct_cached(gauged_coeffs, len(gauged_coeffs), hilbert)
print("Checking reconstruction of original wavefunction: ", np.allclose(recon, wf))
print("Checking reconstruction of gauged wavefunction: ", np.allclose(recond_gauged, wf))



Checking reconstruction of original wavefunction:  True
Checking reconstruction of gauged wavefunction:  True


In [3]:
jax.devices()

[CpuDevice(id=0)]

In [4]:
nspins = 2 
Lambda = float(0.0)
_, hilbert, H = get_TFI_Hamiltonian(nspins, Lambda=Lambda, pbc=False) 

type(H), H.dtype
print(H)

LocalOperatorJax(dim=2, acting_on=[(np.int64(0), np.int64(1))], constant=0.0, dtype=float64)


In [ ]:
vmc_steps = 100
learning_rate = 0.002
diag_shift = float(1e-4)


params = optimization.generate_params(
    alpha=float(1),
    seed=1234,
    learning_rate=float(learning_rate),
    n_iter=int(vmc_steps),
    show_progress=True,
    diag_shift=float(diag_shift),
    Lambda=Lambda,
    out=f"../data/test/test_ising",
)
output_file = params["out"] + ".log"
if os.path.exists(output_file):
    print(f"Skipping Hamiltonian, output file already exists.")

out = optimization.optimize_rbm(H, params)
optimization.write_output(H, out, params, k=1, k_states_save=False)

Skipping Hamiltonian, output file already exists.


  0%|          | 0/100 [00:00<?, ?it/s]

FullSumState(hilbert = Spin(s=1/2, N=2, ordering=new), 


In [12]:
out.parameters
b = out.parameters["Dense"]["bias"]
w = out.parameters["Dense"]["kernel"]
a = out.parameters["visible_bias"]

In [13]:
def c1(x, m):
    ''' Returns the m-th derivative of 1/m! log(cosh(x)) for m = 0, 1, 2, 3, 4'''
    if m == 0:
        return np.log(np.cosh(x))
    if m == 1:
        return np.tanh(x)
    if m == 2:
        return 0.5 * (np.sech(x)**2)
    if m == 3:
        return -1.0/3.0 * np.tanh(x) * (np.sech(x)**2)
    if m == 4:
        return 1.0/12.0 * (np.cosh(2*x) - 2) * (np.sech(x)**4)


In [ ]:
psi = out.to_array()
logpsi = np.log(psi)
cluster_coeffs = fwht_coeffs_in_cluster_col_order(logpsi, hilbert)
print(cluster_coeffs)